In [ ]:
import os
import sys
import argparse
import pandas as pd
import torch
import anndata as ad
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from DeepRUOT.losses import OT_loss1
from DeepRUOT.utils import (
    generate_steps, load_and_merge_config,
    SchrodingerBridgeConditionalFlowMatcher,
    generate_state_trajectory, get_batch, get_batch_size
)
from DeepRUOT.train import train_un1_reduce, train_all
from DeepRUOT.models import FNet_interaction, scoreNet2
from DeepRUOT.constants import DATA_DIR, RES_DIR
from DeepRUOT.exp import setup_exp

### Load config

In [ ]:
config_path = '../config/mosta_config.yaml'

# Load and merge configuration
config = load_and_merge_config(config_path)

### Load data and model

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]
#df = df[df.iloc[:,1] > 0.4]
device = torch.device('cpu')
exp_dir, logger = setup_exp(
            RES_DIR, 
            config, 
            config['exp']['name']
        )
dim = config['data']['dim']

In [ ]:
model_config = config['model']
        
f_net = FNet_interaction(
            in_out_dim=model_config['in_out_dim'],
            hidden_dim=model_config['hidden_dim'],
            n_hiddens=model_config['n_hiddens'],
            activation=model_config['activation'],
            use_spatial = True, 
            num_heads = 8,
            thre = 0.06,
            num_layers = 1,

        ).to(device)

sf2m_score_model = scoreNet2(
    in_out_dim=model_config['in_out_dim'],
    hidden_dim=model_config['score_hidden_dim'],
    activation=model_config['activation']
).float().to(device)

In [ ]:
f_net.load_state_dict(torch.load(os.path.join(exp_dir, 'model_final'),map_location=torch.device('cpu')))
f_net.to(device)
sf2m_score_model.load_state_dict(torch.load(os.path.join(exp_dir, 'score_model'),map_location=torch.device('cpu')))
sf2m_score_model.to(device)

In [ ]:
import scanpy as sc

# 加载 h5ad 文件
adata = sc.read("../spatial_data/Mouse_embryo_all_stage.h5ad")

# 查看数据的基本信息
print(adata)

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance

# 假设 adata 是您的原始 AnnData 对象
# 获取所有唯一的批次名称
batch_names = adata.obs['timepoint'].cat.categories

# 创建字典存储每个批次的 AnnData 对象，确保是实际对象
adata_dict = {}
for batch in batch_names:
    adata_dict[batch] = adata[adata.obs['timepoint'] == batch].copy()

# 定义预处理函数
def preprocess_adata(adata):
    #sc.pp.normalize_total(adata, target_sum=1e4)
    #sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    adata = adata[:, adata.var.highly_variable]
    return adata

# 定义空间坐标缩放函数
def scale_spatial_coords(adata):
    spatial_coords = adata.obsm['spatial']
    x_min, x_max = spatial_coords[:, 0].min(), spatial_coords[:, 0].max()
    y_min, y_max = spatial_coords[:, 1].min(), spatial_coords[:, 1].max()
    x_range = x_max - x_min
    spatial_coords[:, 0] = (spatial_coords[:, 0] - x_min) / x_range
    scale_factor = 1 / x_range
    spatial_coords[:, 1] = (spatial_coords[:, 1] - y_min) * scale_factor
    adata.obsm['spatial'] = spatial_coords # 更新修改后的坐标
    return adata

# 定义绘图函数
def plot_spatial(adata, batch_name):
    spatial_coords = adata.obsm['spatial']
    annotations = adata.obs['annotation']
    annotation_colors = adata.uns['annotation_colors']
    category_to_color = dict(zip(annotations.cat.categories, annotation_colors))
    colors = annotations.map(category_to_color)
    plt.figure(figsize=(10, 8))
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=colors, s=10, alpha=1)
    plt.title(f'Spatial Visualization for {batch_name}')
    plt.xlabel('Scaled X')
    plt.ylabel('Scaled Y')
    plt.show()

def remove_outliers(adata, radius=0.05, threshold=5):
    # 获取空间坐标
    spatial_coords = adata.obsm['spatial']
    # 计算所有数据点之间的距离矩阵
    dist_matrix = distance.cdist(spatial_coords, spatial_coords)
    # 计算每个点的邻居数量（减去自身）
    neighbors = np.sum(dist_matrix < radius, axis=1) - 1
    # 创建掩码：保留邻居数量大于等于阈值的点
    mask = neighbors >= threshold
    # 应用掩码，删除离群点
    adata = adata[mask]
    return adata

# 自动化处理每个批次
for batch in batch_names:
    adata_batch = adata_dict[batch].copy()
    # adata_batch = preprocess_adata(adata_batch)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    # adata_batch = remove_outliers(adata_batch, radius=0.05, threshold=5)
    # adata_batch = scale_spatial_coords(adata_batch.copy())
    adata_dict[batch]=adata_batch.copy()

import numpy as np
import pandas as pd

# 初始化标签列表
labels_list = []
T = 5
batch_indices = [3, 4, 5, 6]

for t, batch_idx in enumerate(batch_indices):
    adata_t = adata_dict[batch_names[batch_idx]]
    labels_t = adata_t.obs['annotation'].values  # 获取当前时间点的 Annotation
    labels_list.append(labels_t)

# 合并所有标签
all_labels = np.concatenate(labels_list)

# 读取 CSV 文件
df_new = pd.read_csv('../data/mosta_four_time.csv')

# 添加 Annotation 列
df_new['Annotation'] = all_labels
# 获取 Annotation 的类别和颜色
if pd.api.types.is_categorical_dtype(adata.obs['annotation']):
    categories = adata.obs['annotation'].cat.categories
else:
    categories = adata.obs['annotation'].unique()  # 如果不是 categorical 类型

colors = adata.uns['annotation_colors']

# 创建标签到颜色的映射
label_to_color = dict(zip(categories, colors))

In [ ]:
df_new

### Add Brain Cell Type

In [ ]:
# 读取带 celltype 的脑数据
adata_brain = sc.read("../spatial_data/adata_brain_with_celltype_5stage.h5ad")

# adata_brain 里的 timepoint 按出现顺序取前4个
brain_batches = adata_brain.obs['timepoint'].cat.categories[:4]
print("Brain 使用的前4个 timepoint:", brain_batches)

# 按顺序提取 celltype
celltype_list = []
for batch in brain_batches:
    adata_b = adata_brain[adata_brain.obs['timepoint'] == batch]
    celltype_list.append(adata_b.obs['celltype'].values)

# 合并所有 celltype
all_celltypes = np.concatenate(celltype_list)

# 找出 df_new 中 Annotation == "Brain" 的位置
brain_mask = df_new['Annotation'] == "Brain"
num_brain = brain_mask.sum()

print("df_new 中 Brain 的数量：", num_brain)
print("提取的 celltype 数量：", len(all_celltypes))

# 检查长度是否一致
if num_brain != len(all_celltypes):
    print("数量不一致，请检查 adata_brain 与 df_new 的对应关系！")
else:
    print("长度一致，将进行合并。")

# 创建 celltype 列（若不存在）
if 'celltype' not in df_new.columns:
    df_new['celltype'] = None

# 仅给 Brain 行填充 celltype
df_new.loc[brain_mask, 'celltype'] = all_celltypes

# 保存结果
df_new.to_csv("../data/mosta_four_time_with_celltype.csv", index=False)

In [ ]:
import pandas as pd

# 读取 CSV 文件
df_new = pd.read_csv("../data/mosta_four_time_with_celltype.csv")
df_new

In [ ]:
import pandas as pd
import numpy as np


# 2. 定义映射字典 (规范化命名)
mapping_dict = {
    # Progenitors (Ancestral states)
    'Forebrain radial glia': 'Apical Progenitors (RG)',
    'Cortical intermediate progenitor': 'Basal Progenitors (IP)',
    
    # Transitioning states
    'Cortical glutamatergic neuroblast': 'Migrating Neuroblasts',
    'Forebrain glutamatergic neuroblast': 'Migrating Neuroblasts',
    'Forebrain neuroblast': 'Migrating Neuroblasts',
    
    # Differentiated Excitatory Neurons (Placeholder for split)
    'Cortical glutamatergic neuron': 'Excitatory Neurons',
    'Cortical or hippocampal glutamatergic neuron': 'Excitatory Neurons',
    
    # Interneurons
    'Forebrain GABAergic neuron': 'Inhibitory Neurons',
    'Forebrain GABAergic neuroblast': 'Inhibitory Neurons',
    
    # Specialized and Glial cells
    'Cajal-Retzius cell': 'Cajal-Retzius Cells',
    'Hindbrain glioblast': 'Glioblasts',
    'Mixed region glioblast': 'Glioblasts',
    'Choroid plexus': 'Choroid Plexus',
    'Choroid plexus progenitor': 'Choroid Plexus'
}

# 3. 执行基础映射
df_new['telencephalon'] = df_new['celltype'].map(mapping_dict).fillna('Other')
df_new
print(df_new['telencephalon'].value_counts())
df_new.to_csv("../data/mosta_four_time_with_celltype_refined.csv", index=False)

### Load Annotated data

In [ ]:
import pandas as pd

# 读取 CSV 文件
df_new = pd.read_csv("../data/mosta_four_time_with_celltype_refined.csv")
df_new

### Plot Velocity

In [ ]:
import numpy as np
import torch
device = 'cpu'
f_net.to(device)
sf2m_score_model.to(device)
# 假设 df, dim, device, f_net 已定义
df= pd.read_csv('../data/mosta_four_time.csv')

all_times = df['samples'].values
all_data = df[[f'x{i}' for i in range(1, dim + 1)]].values

# 转换为 PyTorch 张量
t_tensor = torch.tensor(all_times, dtype=torch.float32).unsqueeze(1).to(device)
data_tensor = torch.tensor(all_data, dtype=torch.float32).to(device)

# 计算 gradients
with torch.no_grad():  # 假设不需要计算梯度以节省内存
    gradients = f_net.v_net(t_tensor, data_tensor)

# 转换为 NumPy 数组
gradients_np = gradients.cpu().numpy()

from DeepRUOT.interaction import cal_interaction

# 获取唯一的时间点
time_points = df['samples'].unique()
all_gradients = []

# 对每个时间点计算梯度
for time in time_points:
    subset = df[df['samples'] == time]
    data = torch.tensor(subset.iloc[:, 1:dim+1].values, dtype=torch.float32).to(device)
    lnw = torch.log(torch.ones(data.shape[0], 1) / data.shape[0]).to(device)
    # 计算交互梯度
    with torch.no_grad():
        gradients_i = cal_interaction(data, lnw, f_net.interaction_net, torch.tensor([time]), m=1024, threshold=1000)
    all_gradients.append(gradients_i.detach().cpu().numpy())

# 拼接所有梯度
gradients_np_retain = np.concatenate(all_gradients, axis=0)

import numpy as np
import matplotlib.pyplot as plt
import torch

# Assume the following are defined:
# - df: DataFrame with columns 'samples', 'x1', 'x2', ..., 'xn'
# - dim: Number of dimensions (n)
# - device: torch.device (e.g., 'cpu' or 'cpu')
# - sf2m_score_model: Your model function that takes t_tensor and data_tensor

# Step 1: Extract all time points and data points


all_times = df['samples'].values
all_data = df[[f'x{i}' for i in range(1, dim + 1)]].values

#t_value = 4.0  
#t_tensor = torch.tensor([t_value] * all_data.shape[0]).unsqueeze(1).float().to(device)
t_tensor = torch.tensor(all_times, dtype=torch.float32).unsqueeze(1).to(device)

# Step 2: Convert to PyTorch tensors
data_tensor = torch.tensor(all_data, dtype=torch.float32).to(device)
data_tensor.requires_grad_(True)  # Enable gradient tracking
#t_tensor = torch.tensor(all_times, dtype=torch.float32).unsqueeze(1).to(device)

# Step 3: Compute log density values and gradients
log_density_values = sf2m_score_model(t_tensor, data_tensor)
log_density_values.backward(torch.ones_like(log_density_values))
gradients_score = data_tensor.grad

In [ ]:
drift = gradients_np + gradients_np_retain + gradients_score.detach().cpu().numpy()

#### VelocityAnalyzer

In [ ]:
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import os
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from scipy.ndimage import gaussian_filter
import scvelo as scv
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def ensure_valid_palette(adata, color_key, provided_palette=None):
    if color_key not in adata.obs.columns:
        return None
    categories = adata.obs[color_key].unique()
    categories = categories[~pd.isna(categories)]
    
    if provided_palette is not None and isinstance(provided_palette, dict):
        missing_keys = [cat for cat in categories if cat not in provided_palette]
        if len(missing_keys) == 0:
            return provided_palette
        else:
            print(f"Warning: Provided palette is missing keys for {color_key}. Generating new palette.")
    
    n_cats = len(categories)
    if n_cats <= 20:
        colors = sc.pl.palettes.vega_20
    elif n_cats <= 28:
        colors = sc.pl.palettes.zeileis_28
    else:
        colors = sc.pl.palettes.godsnot_102
        
    if n_cats > len(colors):
        colors = list(colors) * (n_cats // len(colors) + 1)
        
    new_palette = {cat: colors[i] for i, cat in enumerate(categories)}
    return new_palette

In [ ]:
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import anndata as ad
import seaborn as sns
import torch

def velocity_fingerprint_stream_simple(
    df, f_net, sf2m_score_model, timepoint_idx, dim=52, space='physical',
    velocity_type='full', basis='spatial', density=2, figsize=(12, 10),
    flip_y=True, flip_x=False, n_neighbors=30, mode='default',
    remove_outliers=True, timepoint_str=None, plot_region=None, cell_type=None,
    color='Annotation', palette=None, **kwargs
):
    print(f"Processing timepoint {timepoint_idx} ({space} space, {velocity_type})")
    
    # --------------------------------------------------------------------------
    # 1. 定义 Nature Methods 规范配色 (基于发育谱系色调)
    # --------------------------------------------------------------------------
    TEL_PALETTE = {
        'Apical Progenitors (RG)': '#1f77b4',  # 深蓝
        'Basal Progenitors (IP)': '#aec7e8',   # 浅蓝
        'Immature Neurons': '#2ca02c',         # 绿色
        'Excitatory Neurons': '#ffbb78',  # 橙黄
        'Inhibitory Neurons': '#9467bd',       # 紫色
        'Cajal-Retzius Cells': '#e377c2',      # 粉红
        'Glioblasts': '#8c564b',               # 棕色
        'Choroid Plexus': '#7f7f7f',           # 灰色
        'Other': '#d9d9d9'                     # 浅灰
    }

    # --------------------------------------------------------------------------
    # 2. Data Prep & Background Filter
    # --------------------------------------------------------------------------
    df_t = df[df['samples'] == timepoint_idx].copy()
    
    # 判定上色列：如果指定为 'telencephalon'，则使用该预设列并加载专用颜色
    if color == 'telencephalon':
        color_col_for_adata = 'telencephalon'
        if palette is None:
            palette = TEL_PALETTE
            print("Successfully matched Nature Methods palette for 'telencephalon' column.")
    else:
        color_col_for_adata = color

    # Cell Type Filtering (过滤逻辑)
    if cell_type is not None:
       if 'Annotation' in df_t.columns and cell_type in df_t['Annotation'].values:
            df_t = df_t[df_t['Annotation'] == cell_type].copy()
            print(f"Filter: Found '{cell_type}' in column 'Annotation'")

    # 提取颜色数据
    if color_col_for_adata not in df_t.columns:
        obs_color = ['Unspecified'] * len(df_t)
        df_t['_temp_color'] = 'Unspecified'
        color_col_for_adata = '_temp_color'
    else:
        obs_color = df_t[color_col_for_adata].values

    # Extract Data
    all_data = df_t[[f'x{i}' for i in range(1, dim + 1)]].values
    coords = all_data[:, 0:2]  
    X_expression = all_data[:, 2:] 
    
    # Neural Network (Mocked for context, use your real calc)
    device = 'cpu'
    t_tensor = torch.full((all_data.shape[0], 1), fill_value=timepoint_idx, dtype=torch.float32, device=device)
    data_tensor = torch.tensor(all_data, dtype=torch.float32, device=device)
    with torch.no_grad():
        drift_full = f_net.v_net(t_tensor, data_tensor).detach().cpu().numpy()
    lnw = torch.log(torch.ones(data_tensor.shape[0], 1, device=device) / data_tensor.shape[0])
    with torch.no_grad():
        interaction_full = cal_interaction(
            data_tensor, lnw, f_net.interaction_net,
            torch.tensor([timepoint_idx], dtype=torch.float32, device=device),
            m=1024, threshold=1000
        ).detach().cpu().numpy()
        
    V3_intr = drift_full[:, 0:2]           
    V1_rna_intr = drift_full[:, 2:]        
    V4_inter = interaction_full[:, 0:2]    
    V2_rna_inter = interaction_full[:, 2:] 

    if X_expression.shape[1] != V1_rna_intr.shape[1]:
        raise ValueError(f"Shape Mismatch!")

    if space == "physical":
        V_intr = V3_intr
        V_inter = V4_inter
        space_title = "Physical Space"
    elif space == "gene":
        print(f"  Gene space: Projecting high-dim velocity to 2D using scVelo...")
        space_title = "Gene Space"
        def run_scvelo_projection(V_high_dim, tag):
            tmp_ad = ad.AnnData(X=X_expression)
            tmp_ad.layers['Ms'] = X_expression.copy()
            tmp_ad.layers['velocity'] = V_high_dim.copy()
            tmp_ad.obsm['X_spatial'] = coords.copy()
            sc.pp.neighbors(tmp_ad, n_neighbors=n_neighbors, use_rep='X')
            scv.tl.velocity_graph(tmp_ad, vkey='velocity', xkey='Ms', n_jobs=-1)
            scv.tl.velocity_embedding(tmp_ad, basis='spatial', vkey='velocity')
            return tmp_ad.obsm['velocity_spatial']
        print("  - Projecting Intrinsic component...")
        V_intr = run_scvelo_projection(V1_rna_intr, "Intrinsic")
        print("  - Projecting Interaction component...")
        V_inter = run_scvelo_projection(V2_rna_inter, "Interaction")
    else:
        raise ValueError(f"Unknown space: {space}")

    # 1. 检查并修复 V_intr (Intrinsic Velocity)
    if not np.isfinite(V_intr).all():
        nan_count = np.sum(~np.isfinite(V_intr))
        print(f"Warning: Found {nan_count} non-finite values in Intrinsic Velocity. Replacing with 0.")
        # 将 NaN 和 Inf 替换为 0，防止 PDF 保存报错
        V_intr = np.nan_to_num(V_intr, nan=0.0, posinf=0.0, neginf=0.0)

    # 2. 检查并修复 V_inter (Interaction Velocity)
    if not np.isfinite(V_inter).all():
        nan_count = np.sum(~np.isfinite(V_inter))
        print(f"Warning: Found {nan_count} non-finite values in Interaction Velocity. Replacing with 0.")
        V_inter = np.nan_to_num(V_inter, nan=0.0, posinf=0.0, neginf=0.0)
        
    # 3. 检查并修复坐标 (虽然较少见，但为了保险)
    if not np.isfinite(coords).all():
        print(f"Warning: Found non-finite values in Spatial Coords. Cleaning...")
        coords = np.nan_to_num(coords, nan=0.0, posinf=0.0, neginf=0.0)    

    adata = ad.AnnData(X=all_data) 
    adata.obsm["X_spatial"] = coords
    adata.obsm["velocity_intrinsic_spatial"] = V_intr
    adata.obsm["velocity_interaction_spatial"] = V_inter
    
    adata.obs[color_col_for_adata] = obs_color
    adata.obs[color_col_for_adata] = adata.obs[color_col_for_adata].astype('category')
    
    # Auto Palette
    final_palette = ensure_valid_palette(adata, color_col_for_adata, palette)
    
    # Plotting
    fig1, ax1 = plot_single_velocity_field(
        adata, "velocity_intrinsic", density, figsize, flip_y, flip_x, 
        f"{space_title} - Intrinsic Velocity", color_col_for_adata, mode, 
        remove_outliers, timepoint_str, plot_region, final_palette, **kwargs 
    )
    
    fig2, ax2 = plot_single_velocity_field(
        adata, "velocity_interaction", density, figsize, flip_y, flip_x, 
        f"{space_title} - Interaction Velocity", color_col_for_adata, mode, 
        remove_outliers, timepoint_str, plot_region, final_palette, **kwargs 
    )
    
    return adata, [ax1, ax2], [fig1, fig2]

In [ ]:
def plot_single_velocity_field(adata, velocity_key, density, figsize, flip_y, flip_x, title, color_key, mode='default', remove_outliers=True, timepoint_str=None, plot_region=None, palette=None, **kwargs):
    alpha_val = kwargs.get('alpha', 0.25)
    
    if mode == 'black':
        plt.style.use('dark_background')
        background_color, text_color = 'black', 'white'
    else:
        plt.style.use('default')
        background_color, text_color = 'white', 'black'
        
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    # 1. 数据裁切 (Subset Data)
    # 这让 scVelo 的流线只在局部区域计算，细节更好
    adata_plot = adata.copy()
    if remove_outliers:
        y = adata_plot.obsm['X_spatial'][:, 1]
        Q1, Q3 = np.percentile(y, [25, 75])
        IQR = Q3 - Q1
        mask = (y >= Q1 - 1.5*IQR) & (y <= Q3 + 1.5*IQR)
        adata_plot = adata_plot[mask].copy()

    if plot_region is not None:
        x_min, x_max, y_min, y_max = plot_region
        X = adata_plot.obsm['X_spatial']
        mask = np.ones(len(X), dtype=bool)
        if x_min is not None: mask &= (X[:, 0] > x_min)
        if x_max is not None: mask &= (X[:, 0] < x_max)
        if y_min is not None: mask &= (X[:, 1] > y_min)
        if y_max is not None: mask &= (X[:, 1] < y_max)
        adata_plot = adata_plot[mask].copy()
        print(f"  Zoom-in subset: {len(adata_plot)} cells remaining.")

    point_size = 50 if plot_region is None else 60
    
    scv.pl.velocity_embedding_stream(
        adata_plot, basis='spatial', vkey=velocity_key, color=color_key, palette=palette,
        ax=ax, show=False, density=density, smooth=0.8, min_mass=1, cutoff_perc=3, linewidth=1.5,
        arrow_size=1.2, alpha=alpha_val, size=point_size, legend_loc='right margin', title='', frameon=False
    )

    if flip_y: ax.invert_yaxis()
    if flip_x: ax.invert_xaxis()
    
    # 2. 视图锁定 (Lock View)
    # 强制 Matplotlib 锁定在用户指定的区域，即使有些流线画到了外面
    if plot_region is not None:
        x_min, x_max, y_min, y_max = plot_region
        # 获取当前 limit，如果用户传了 None 则保持当前 limit
        cur_xlim = ax.get_xlim()
        cur_ylim = ax.get_ylim()
        
        # 翻转逻辑 check：如果 flip_y 为 True，y轴是倒置的 (大值在下)
        # set_ylim 需要根据当前轴的方向来设置
        
        ax.set_xlim(
            x_min if x_min is not None else cur_xlim[0],
            x_max if x_max is not None else cur_xlim[1]
        )
        
        # 对于 Y 轴，如果翻转了，要注意 min/max 的顺序
        target_ymin = y_min if y_min is not None else (cur_ylim[0] if not flip_y else cur_ylim[1])
        target_ymax = y_max if y_max is not None else (cur_ylim[1] if not flip_y else cur_ylim[0])
        
        # 简单处理：直接 set_ylim，matplotlib 会自动处理翻转轴的数值大小顺序
        # 但为了安全，我们还是尊重用户输入的数值意义 (数值上的 min 和 max)
        
        if flip_y:
            ax.set_ylim(target_ymax, target_ymin) # 倒置：大值在下(bottom), 小值在上(top)
        else:
            ax.set_ylim(target_ymin, target_ymax)

    full_title = f"{title} - {timepoint_str}" if timepoint_str else title
    ax.set_title(full_title, fontsize=20, fontweight="bold", color=text_color, pad=20)
    for spine in ax.spines.values(): spine.set_color(text_color)
    ax.tick_params(colors=text_color, labelsize=12)
    ax.xaxis.label.set_color(text_color)
    ax.yaxis.label.set_color(text_color)
    
    return fig, ax

In [ ]:
class VelocityAnalyzer:
    def __init__(self, df, f_net, sf2m_score_model, dim=52):
        self.df = df
        self.f_net = f_net
        self.sf2m_score_model = sf2m_score_model
        self.dim = dim
        if 'Annotation' not in df.columns:
            print("Warning: 'Annotation' column not found.")

    def plot_fingerprint(
            self,
            adata,
            timepoint,
            timepoint_str,
            all_time_communication,
            space='physical',
            cell_type="celltype",        
            background_cell_type=None,   
            save_path=None,
            density=2,
            figsize=(7, 5),
            n_neighbors=30,
            mode='default',
            remove_outliers=True,
            plot_region=None,
            flip_x=False,
            communication=True,
            color='Annotation',
            label_to_color=None,
            **kwargs
        ):
        # 处理 Communication Focus
        target_focus_cell = cell_type 
        if target_focus_cell == "celltype": 
            target_focus_cell = None
        
        target_background_cell = background_cell_type
        print(f"\n=== Plotting Velocity (Bg: {target_background_cell}) & Comm (Focus: {target_focus_cell}) ===")
        
        bg_alpha = 0.35 if communication else 0.4

        # 1. Base Plot
        ad_res, axes, figs = velocity_fingerprint_stream_simple(
            df=self.df, f_net=self.f_net, sf2m_score_model=self.sf2m_score_model,
            timepoint_idx=timepoint, dim=52, space=space, basis='spatial',
            density=density, figsize=figsize, flip_y=False, flip_x=flip_x,
            n_neighbors=n_neighbors, cell_type=target_background_cell,
            mode=mode, remove_outliers=remove_outliers, timepoint_str=timepoint_str,
            plot_region=plot_region, color=color, palette=label_to_color, alpha=bg_alpha, **kwargs
        )

        # 2. Overlay Communication Edges
        if communication and all_time_communication is not None:
            comm_data = all_time_communication.get(timepoint_str)
            if isinstance(comm_data, list) and len(comm_data) > 0: comm_data = comm_data[0]
            
            matrix = comm_data.get('M_per_source') if comm_data else None
            
            if matrix is not None and 'types' in comm_data:
                types = comm_data['types']
                comm_df = pd.DataFrame(matrix, index=types, columns=types)
                
                # 筛选有效节点
                valid_nodes = [ct for ct in comm_df.index if ct in self.df['Annotation'].unique()]
                if target_focus_cell is not None and target_focus_cell in comm_df.index:
                    weight_thresh = 1
                    keep_nodes = {target_focus_cell}
                    for ct in valid_nodes:
                        if ct == target_focus_cell: continue
                        if max(float(comm_df.loc[target_focus_cell, ct]), float(comm_df.loc[ct, target_focus_cell])) > weight_thresh:
                            keep_nodes.add(ct)
                    valid_nodes = [ct for ct in valid_nodes if ct in keep_nodes]

                # --- 核心逻辑修改：区分 Brain 和其他节点的坐标 ---
                custom_centroids = {}
                time_df = self.df[self.df['samples'] == timepoint]
                
                for ct in valid_nodes:
                    ct_subs = time_df[time_df['Annotation'] == ct]
                    if len(ct_subs) == 0: continue
                    
                    # 如果是 Brain (大脑)，保持原有的全质心计算
                    if ct == "Brain":
                        custom_centroids[ct] = ct_subs[['x1', 'x2']].mean().values
                    else:
                        # 与大脑通信的其他节点，取 y轴(x2) 最大的前100个点的均值
                        top_cells = ct_subs.nlargest(min(len(ct_subs), 100), 'x2')
                        custom_centroids[ct] = top_cells[['x1', 'x2']].mean().values
                
                valid_nodes = [ct for ct in valid_nodes if ct in custom_centroids]
                # ----------------------------------------------

                if plot_region:
                    zoom_xmin, zoom_xmax, zoom_ymin, zoom_ymax = plot_region
                else:
                    zoom_xmin, zoom_xmax, zoom_ymin, zoom_ymax = None, None, None, None

                for ax in axes:
                    drawn_nodes = set()
                    for src in valid_nodes:
                        for tgt in valid_nodes:
                            weight = float(comm_df.loc[src, tgt])
                            if weight <= 0: continue 
                            if target_focus_cell and (src != target_focus_cell and tgt != target_focus_cell):
                                continue

                            p1, p2 = custom_centroids[src], custom_centroids[tgt]
                            if not (np.isfinite(p1).all() and np.isfinite(p2).all()): continue

                            # 区域过滤逻辑
                            def is_in_region(p):
                                if zoom_xmin is not None and (p[0] < zoom_xmin or p[0] > zoom_xmax): return False
                                if zoom_ymin is not None and (p[1] < zoom_ymin or p[1] > zoom_ymax): return False
                                return True

                            if not (is_in_region(p1) and is_in_region(p2)): continue
                                
                            max_weight = comm_df.loc[valid_nodes, valid_nodes].values.max()
                            lw = 1 + 10 * np.sqrt(weight / max_weight)            
                            edge_color = label_to_color[src] if (label_to_color and src in label_to_color) else '#333333'

                            # 自环逻辑 (针对 Brain 的自环由于 p1 没变，效果将保持原样)
                            if src == tgt:
                                start = (p1[0] - 0.03, p1[1] + 0.05)
                                end   = (p1[0] + 0.06, p1[1] + 0.02)
                                rad, curr_mutation_scale = -10.0, 8
                            else:
                                start, end = (p1[0], p1[1]), (p2[0], p2[1])
                                rad, curr_mutation_scale = 0.15, 20
                    
                            arrow = mpatches.FancyArrowPatch(
                                start, end, arrowstyle='-|>,head_length=0.8,head_width=0.5',
                                mutation_scale=curr_mutation_scale, connectionstyle=f"arc3,rad={rad}",
                                color=edge_color, linewidth=lw, alpha=0.9, zorder=30 
                            )
                            ax.add_patch(arrow)
                            
                            # 绘制节点 scatter
                            for node_name, pos in [(src, p1), (tgt, p2)]:
                                if node_name not in drawn_nodes:
                                    node_c = label_to_color[node_name] if (label_to_color and node_name in label_to_color) else 'grey'
                                    ax.scatter(pos[0], pos[1], s=2000, c=node_c, edgecolors='white', linewidth=2.5, zorder=31)
                                    drawn_nodes.add(node_name)
                                        
            
        # 3. Save
        if save_path:
            base = save_path.replace('.pdf', '').replace('.png', '')
            
            # 文件名逻辑：区分 background 和 communication focus
            bg_s = f"_bg-{target_background_cell}" if target_background_cell else ""
            focus_s = f"_focus-{target_focus_cell}" if target_focus_cell else ""
            
            # 添加 zoom 标记到文件名
            zoom_s = "_zoom" if plot_region else ""
            import os
            os.makedirs(os.path.dirname(base), exist_ok=True)
            
            # 文件名例如: filename_bg-Brain_focus-Brain_intrinsic.pdf
            figs[0].savefig(f"{base}{bg_s}{focus_s}{zoom_s}_intrinsic.pdf", dpi=300, bbox_inches='tight')
            figs[1].savefig(f"{base}{bg_s}{focus_s}{zoom_s}_interaction.pdf", dpi=300, bbox_inches='tight')
            print(f"Saved figures to {base}{bg_s}{focus_s}{zoom_s}_*.pdf")

        return ad_res, axes, figs
        
    def _save_isolated_legend(self, adata, color_key, save_path):
        """
        内部私有方法：根据 adata 实际颜色映射保存 Legend PDF
        """
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches

        # 检查 adata 是否包含颜色信息
        color_uns_key = f"{color_key}_colors"
        if color_uns_key not in adata.uns:
            return

        # 提取当前图中存在的类别和对应的颜色
        categories = adata.obs[color_key].cat.categories
        colors = adata.uns[color_uns_key]
        present_labels = adata.obs[color_key].unique()

        # 创建一个不显示在 notebook 里的独立 figure
        fig_leg = plt.figure(figsize=(3, len(present_labels) * 0.4))
        ax_leg = fig_leg.add_subplot(111)
        ax_leg.axis('off')

        patches = [
            mpatches.Patch(color=c, label=cat) 
            for cat, c in zip(categories, colors) if cat in present_labels
        ]

        ax_leg.legend(
            handles=patches, loc='center', frameon=False, 
            fontsize=10, handlelength=1.5, labelspacing=0.8
        )

        # 构造 legend 文件名
        leg_base = save_path.replace('.pdf', '').replace('.png', '')
        leg_path = f"{leg_base}_legend.pdf"
        
        # 使用 bbox_inches='tight' 确保 PDF 不会有大片留白
        fig_leg.savefig(leg_path, transparent=True, bbox_inches='tight')
        plt.close(fig_leg) # 关键：立即关闭，不在单元格展示
        print(f"Isolated legend saved to: {leg_path}")
            

In [ ]:
# 初始化
analyzer = VelocityAnalyzer(df_new, f_net, sf2m_score_model, dim=52)

In [ ]:
adata_dict

In [ ]:
df_new

In [ ]:
adata_brain = adata[adata.obs['annotation'] == 'Brain'].copy()

In [ ]:
adata_brain.obsm['spatial'][:, :2]

### Communication

In [ ]:
import pickle
with open("all_timepoint_communications_merged.pkl", "rb") as f:
    all_time_communications = pickle.load(f)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E12.5"],
    timepoint=0,
    timepoint_str="E12.5",
    space='gene',
    color='Annotation',
    mode='default',
    cell_type="Brain",
    communication=True,
    figsize=(16, 20),
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_communication_gene_brain_t0_combined.pdf',
    label_to_color=label_to_color,
    density=2
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E13.5"],
    timepoint=1,
    timepoint_str="E13.5",
    space='gene',
    color='Annotation',
    mode='default',
    cell_type="Brain",
    communication=True,
    figsize=(16, 20),
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_communication_gene_brain_t1_combined.pdf',
    label_to_color=label_to_color,
    density=2
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E14.5"],
    timepoint=2,
    timepoint_str="E14.5",
    space='gene',
    color='Annotation',
    mode='default',
    cell_type="Brain",
    communication=True,
    figsize=(16, 20),
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_communication_gene_brain_t2_combined.pdf',
    label_to_color=label_to_color,
    density=2
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E15.5"],
    timepoint=3,
    timepoint_str="E15.5",
    space='physical',
    color='Annotation',
    mode='default',
    cell_type="Brain",
    communication=True,
    figsize=(16, 20),
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_communication_physical_brain_t3_combined.pdf',
    label_to_color=label_to_color,
    density=2
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E15.5"],
    timepoint=3,
    timepoint_str="E15.5",
    space='gene',
    color='Annotation',
    mode='default',
    cell_type="Brain",
    communication=True,
    figsize=(16, 20),
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_communication_gene_brain_t3_combined.pdf',
    label_to_color=label_to_color,
    density=2
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E12.5"],
    timepoint=0,
    timepoint_str="E12.5",
    space='physical',
    color='celltype',
    mode='default',
    cell_type="Brain",    # 只显示与 Brain 相关的连线
    background_cell_type="Brain", # 整个图只画 Brain 区域的流线和散点
    figsize=(17.5, 12.5),
    communication=False,
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_physical_brain_t3_combined.pdf',
    label_to_color=label_to_color,
    density=4
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E15.5"],
    timepoint=3,
    timepoint_str="E15.5",
    space='gene',
    color='celltype',
    mode='default',
    cell_type="Brain",    # 只显示与 Brain 相关的连线
    background_cell_type="Brain", # 整个图只画 Brain 区域的流线和散点
    figsize=(17.5, 12.5),
    communication=False,
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_gene_brain_t3_combined.png',
    label_to_color=label_to_color,
    density=1.5
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E15.5"],
    timepoint=3,
    timepoint_str="E15.5",
    space='physical',
    color='telencephalon',
    mode='default',
    cell_type="Brain",    # 只显示与 Brain 相关的连线
    background_cell_type="Brain", # 整个图只画 Brain 区域的流线和散点
    plot_region=[-1.3, -0.5, 3.3, 4.2],
    communication=False,
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_physical_telencephalon_t3_combined.pdf',
    label_to_color=label_to_color,
    density=1
)

In [ ]:
analyzer.plot_fingerprint(
    adata_dict["E15.5"],
    timepoint=3,
    timepoint_str="E15.5",
    space='gene',
    color='telencephalon',
    mode='default',
    cell_type="Brain",    # 只显示与 Brain 相关的连线
    background_cell_type="Brain", # 整个图只画 Brain 区域的流线和散点
    plot_region=[-1.3, -0.5, 3.3, 4.2],
    communication=False,
    all_time_communication=all_time_communications,
    save_path=f'{exp_dir}/velocity_gene_telencephalon_t3_combined.pdf',
    label_to_color=label_to_color,
    density=0.75
)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def save_telencephalon_legend(save_path="telencephalon_legend.pdf"):

    TEL_PALETTE = {
        'Apical Progenitors (RG)': '#1f77b4',  # 深蓝
        'Basal Progenitors (IP)': '#aec7e8',   # 浅蓝
        'Immature Neurons': '#2ca02c',         # 绿色
        'Excitatory Neurons': '#ffbb78',  # 橙黄
        'Inhibitory Neurons': '#9467bd',       # 紫色
        'Cajal-Retzius Cells': '#e377c2',      # 粉红
        'Glioblasts': '#8c564b',               # 棕色
        'Choroid Plexus': '#7f7f7f',           # 灰色
        'Other': '#d9d9d9'                     # 浅灰
    }

    # 2. 创建一个没有坐标轴的空白画布
    # figsize 可以根据标签数量调整
    fig = plt.figure(figsize=(3, 4)) 
    ax = fig.add_subplot(111)
    ax.axis('off')

    # 3. 创建 Patch 句柄
    patches = [
        mpatches.Patch(color=color, label=label) 
        for label, color in TEL_PALETTE.items()
    ]

    # 4. 绘制图例
    # frameon=False 符合顶刊简洁审美
    legend = ax.legend(
        handles=patches, 
        loc='center', 
        frameon=False,
        fontsize=10,
        handlelength=1.5, # 调整色块的长宽比
        handleheight=1.0,
        labelspacing=0.8  # 调整行间距
    )

    # 5. 自动调整并保存为矢量 PDF
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', transparent=True)
    plt.close()
    print(f"Legend saved successfully to: {save_path}")

# 执行保存
matplotlib.rcParams['pdf.fonttype'] = 42 # 确保文字作为字体保存，而不是轮廓
save_telencephalon_legend("telencephalon_legend_for_publication.pdf")